# 03 — Gate 2: one-detector probe (e5 + XGBoost), direct -> indirect
**v2: realistic indirect target.** Indirect attacks are now a benign BIPIA document (email/table) with a text-attack instruction injected, plus clean documents as benigns -- a true instruction-in-document distribution, not bare attack strings.

Train on **direct** (deepset), freeze the operating threshold on the source, apply unchanged to the **indirect** distribution, and read S/FNCR/ECE_atk.

**Question:** does miss-severity S stay catastrophic on REAL indirect samples while AUROC roughly holds? **GPU recommended.**

## Session bootstrap

In [10]:
# === SESSION BOOTSTRAP (run first, every session) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
print('Session ready - identity set, src/ on path, repo pulled.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/PICALIB_Research/picalib-research
Already up to date.
Session ready - identity set, src/ on path, repo pulled.


## Dependencies + device

In [11]:
!pip install -q sentence-transformers xgboost scikit-learn datasets
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

CUDA: True | Tesla T4


## Load source (deepset direct) + realistic indirect target (BIPIA)
First run computes + caches embeddings; re-runs reload. Note the new cache file `emb_bipia_indirect.npy` (distinct from the old bare-string cache).

In [ ]:
import importlib, data_loaders, detectors, metrics
for m in (data_loaders, detectors, metrics): importlib.reload(m)
from data_loaders import load_deepset, load_bipia_indirect
from detectors import embed_e5, train_xgb, predict_p, threshold_at_fpr, auroc
from metrics import all_metrics, severity_S, bootstrap_ci
import numpy as np, pandas as pd, os
from sklearn.model_selection import train_test_split

deepset = load_deepset()                     # direct + benign (SOURCE)
bipia   = load_bipia_indirect(n_attack=400)  # realistic indirect (TARGET)
print('deepset', len(deepset),
      '| bipia indirect', len(bipia),
      '(attack', int((bipia.label==1).sum()), '/ benign', int((bipia.label==0).sum()), ')')

os.makedirs('data', exist_ok=True)
def cached_embed(texts, path):
    if os.path.exists(path):
        print('load cached', path); return np.load(path)
    emb = embed_e5(list(texts)); np.save(path, emb); return emb
E_deepset = cached_embed(deepset.text, 'data/emb_deepset.npy')
E_bipia   = cached_embed(bipia.text,   'data/emb_bipia_indirect.npy')
print('emb', E_deepset.shape, E_bipia.shape)

BIPIA indirect: 1097 contexts, 150 text attacks
deepset 662 | bipia indirect 1177 (attack 399 / benign 778 )
load cached data/emb_deepset.npy


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

## Train on direct, freeze threshold at 1% FPR on source
Source-threshold transfer (cite Gate AI 2606.02959). *Probe caveat: threshold in-sample on train benigns; final uses out-of-fold.*

In [ ]:
idx = np.arange(len(deepset))
tr, te = train_test_split(idx, test_size=0.3, stratify=deepset.label, random_state=0)
Xtr, ytr = E_deepset[tr], deepset.label.values[tr]
Xte, yte = E_deepset[te], deepset.label.values[te]
clf = train_xgb(Xtr, ytr, seed=0)
p_tr = predict_p(clf, Xtr)
t = threshold_at_fpr(p_tr[ytr==0], target_fpr=0.01)
print(f'frozen source threshold t = {t:.4f}')

## Source (direct) vs target (realistic indirect) at the FROZEN threshold

In [ ]:
p_s = predict_p(clf, Xte)
m_s = all_metrics(yte, p_s, t=t); m_s['AUROC'] = auroc(yte, p_s)

y_tgt = bipia.label.values            # BIPIA's OWN attacks + benigns (fully indirect)
p_t = predict_p(clf, E_bipia)
m_t = all_metrics(y_tgt, p_t, t=t); m_t['AUROC'] = auroc(y_tgt, p_t)

rows = ['AUROC','FNR','S','FNCR','HCFN','ECE_atk','ECE_pooled']
cmp = pd.DataFrame({'source(direct)':[m_s[k] for k in rows],
                    'target(indirect)':[m_t[k] for k in rows]}, index=rows).round(4)
print(f'frozen t = {t:.4f}  | source n_attack={m_s["n_attacks"]} (misses {m_s["n_misses"]}),'
      f' target n_attack={m_t["n_attacks"]} (misses {m_t["n_misses"]})\n')
print(cmp.to_string())

## Bootstrap 95% CIs on S

In [ ]:
s_pt, s_lo, s_hi = bootstrap_ci(severity_S, yte, p_s, n_boot=1000, t=t)
t_pt, t_lo, t_hi = bootstrap_ci(severity_S, y_tgt, p_t, n_boot=1000, t=t)
print(f'S source {s_pt} [{s_lo:.3f}, {s_hi:.3f}]  (NaN = too few misses to estimate)')
print(f'S target {t_pt:.3f} [{t_lo:.3f}, {t_hi:.3f}]')

## Figure + save report

In [ ]:
import matplotlib.pyplot as plt
keys = ['FNR','S','FNCR','ECE_atk']
xs = np.arange(len(keys)); w = 0.35
fig, ax = plt.subplots(figsize=(6,4))
sv = [0 if (isinstance(m_s[k],float) and m_s[k]!=m_s[k]) else m_s[k] for k in keys]
ax.bar(xs-w/2, sv, w, label='source (direct)')
ax.bar(xs+w/2, [m_t[k] for k in keys], w, label='target (indirect)')
ax.set_xticks(xs); ax.set_xticklabels(keys); ax.set_ylabel('value')
ax.set_title('Gate 2 v2: direct -> realistic indirect at frozen threshold'); ax.legend()
plt.tight_layout(); os.makedirs('figures', exist_ok=True)
plt.savefig('figures/gate2_source_vs_target.png', dpi=150); plt.show()
os.makedirs('reports', exist_ok=True)
cmp.to_csv('reports/gate2_source_vs_target.csv')
print('saved figure + reports/gate2_source_vs_target.csv')

## Verdict

In [ ]:
dA = m_s['AUROC'] - m_t['AUROC']
Stgt = m_t['S']
print('INTERPRETATION')
print(f"  AUROC : {m_s['AUROC']:.3f} -> {m_t['AUROC']:.3f}  (drop {dA:.3f})")
print(f"  S     : {m_s['S']} -> {Stgt}")
print(f"  FNR   : {m_s['FNR']:.3f} -> {m_t['FNR']:.3f}")
print(f"  FNCR  : {m_s['FNCR']:.3f} -> {m_t['FNCR']:.3f}")
if (not np.isnan(Stgt)) and Stgt > 0.5 and m_t['FNCR'] > m_s['FNCR']:
    print('\n  -> CORE EFFECT HOLDS on realistic indirect samples:')
    print('     missed indirect attacks are waved through with high confidence (S high).')
    if dA < 0.15:
        print('     AUROC roughly holds -> calibration/operating-point failure. SCALE UP.')
    else:
        print('     AUROC also drops -> mixed discrimination+calibration failure (still real).')
else:
    print('\n  -> Severity did NOT hold on realistic indirect. The earlier S=0.99 was')
    print('     likely an artefact of bare attack strings. Reassess before the full panel.')

---
## Commit + push

In [ ]:
!git add -A
!git commit -m "Gate 2 v2: realistic indirect (BIPIA context-assembled) direct->indirect transfer"
!git push
print('Pushed. Confirm GitHub + Drive in sync.')